In [13]:
from IPython.display import clear_output
!pip install gradio requests
clear_output()

In [ ]:
import gradio as gr
import requests

# Insert your Groq API key here
GROQ_API_KEY = "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"

GROQ_API_KEY = "https://api.groq.com/openai/v1/chat/completions"

In [ ]:
def respond(
    message,
    history,
    system_message,
    max_tokens,
    temperature,
    top_p,
):
    messages = [{"role": "system", "content": system_message}]
    
    for user, assistant in history:
        if user:
            messages.append({"role": "user", "content": user})
        if assistant:
            messages.append({"role": "assistant", "content": assistant})
    
    messages.append({"role": "user", "content": message})

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "llama-3.3-70b-versatile",
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "top_p": top_p
    }

    response = requests.post(GROQ_API_KEY, headers=headers, json=data)

    if response.status_code != 200:
        return "Error: Unable to get response from model."

    reply = response.json()['choices'][0]['message']['content']

    # Add professional disclaimer
    disclaimer = "\n\nDisclaimer: This information is for educational purposes only and does not substitute professional medical advice. Please consult a qualified healthcare provider for diagnosis or treatment."

    return reply.strip() + disclaimer

In [16]:
demo = gr.ChatInterface(
    respond,
    additional_inputs=[
        gr.Textbox(
            value=(
                "You are an advanced Medical AI Assistant.\n\n"
                "Provide responses in the following structured format:\n"
                "1. Possible Causes\n"
                "2. Symptoms Explanation\n"
                "3. Recommended Actions\n"
                "4. When to Consult a Doctor\n\n"
                "Ensure clarity, medical relevance, and safety. Avoid speculation."
            ),
            label="System Instruction"
        ),
        gr.Slider(100, 1000, value=300, step=50, label="Max Tokens"),
        gr.Slider(0.1, 1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(0.1, 1.0, value=0.9, step=0.05, label="Top-p"),
    ],
    title="Medical AI Assistant",
    description=(
        "An advanced AI-powered medical assistant that provides structured, "
        "informative responses based on user queries."
    ),
    examples=[
        ["I have a headache and fever. What should I do?"],
        ["What are early symptoms of diabetes?"],
        ["How can I improve my sleep quality?"]
    ],
)

In [17]:
if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
